# V2-pattern: baseline + live context

SerpAPI supplies a typical weekly baseline; cached Weather, GBFS, MTA and active reports adjust the final prediction.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
import db_utils

HERE = Path.cwd()
SNAPSHOT = HERE / 'output/serpapi_repeat_audit/snapshots/20260716T103617Z/popular_times_snapshot.csv'
LABEL_DIR = HERE / 'output/serpapi_v2_labels_20260716'
LEGACY_LABELS = HERE / 'output/serpapi_repeat_audit/legacy_cached_baseline.csv'
MODEL_DIR = HERE / 'output/v2_pattern_known_venue_serpapi_20260716'
assert SNAPSHOT.exists() and LEGACY_LABELS.exists(), 'Missing current snapshot or legacy temporal baseline'

In [2]:
# 1. SerpAPI snapshot → weak labels
subprocess.run([
    sys.executable, 'serpapi_popular_times_labels.py',
    '--snapshot', str(SNAPSHOT), '--output-dir', str(LABEL_DIR),
], cwd=HERE, check=True)
LABELS = LABEL_DIR / 'serpapi_popular_times_weak_labels.csv'

{
  "target_type": "google_popular_times_proxy",
  "label_provenance": "serpapi_google_maps_place_snapshot",
  "source_snapshot": "/Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/7.13-7.18/output/serpapi_repeat_audit/snapshots/20260716T103617Z/popular_times_snapshot.csv",
  "snapshot_ids": [
    "20260716T103617Z"
  ],
  "label_rows": 20145,
  "venues": 162,
  "places": 116,
  "score_min": 0,
  "score_max": 100
}


In [3]:
# 2. Input gate — inspect coverage before training.
labels = pd.read_csv(LABELS, dtype={'venue_id': str, 'place_id': str})
venue_ids = labels['venue_id'].drop_duplicates().tolist()
marks = ','.join(['%s'] * len(venue_ids))
venue_inputs = db_utils.read_sql(
    'SELECT venue_id, venue_type, district, rating, latitude, longitude, weather_risk, source_confidence, accessible_status, active_warning, open_now FROM venues WHERE venue_id IN (' + marks + ')',
    tuple(venue_ids),
)
training_inputs = labels.merge(venue_inputs, on='venue_id', how='left')
training_inputs['is_weekend'] = (training_inputs['day_of_week'] >= 5).astype(int)
baseline_fields = ['day_of_week', 'hour_of_day', 'is_weekend', 'venue_type', 'district', 'rating']
enriched_only_fields = ['venue_id', 'latitude', 'longitude', 'weather_risk', 'source_confidence', 'accessible_status', 'active_warning', 'open_now', 'rating_missing']
training_inputs['rating_missing'] = training_inputs['rating'].isna().astype(int)
dynamic_fields = [
    'temperature_c', 'precipitation_mm', 'relative_humidity_pct', 'wind_speed_kmh', 'heat_alert',
    'citibike_station_activity', 'nearby_bike_availability', 'nearby_dock_availability',
    'mta_service_disruption_flag', 'mta_realtime_arrival_count_1h',
    'recent_report_count_1h', 'recent_report_count_3h', 'crowding_report_count_3h',
]
feature_summary = pd.DataFrame([
    {'block': 'learned baseline', 'raw_features': len(baseline_fields), 'model_features_after_encoding': 12},
    {'block': 'venue-specific enrichment', 'raw_features': len(baseline_fields) + len(enriched_only_fields), 'model_features_after_encoding': 'venue_id + static attributes'},
    {'block': 'dynamic serving context', 'raw_features': len(dynamic_fields), 'model_features_after_encoding': 'rule/residual layer'},
    {'block': 'final prediction inputs', 'raw_features': len(baseline_fields) + len(enriched_only_fields) + len(dynamic_fields), 'model_features_after_encoding': 'enriched baseline + dynamic_delta'},
])
static_coverage = pd.DataFrame({'feature': baseline_fields + enriched_only_fields, 'coverage_pct': [round(training_inputs[f].notna().mean() * 100, 2) for f in baseline_fields + enriched_only_fields]})
cache_status = db_utils.read_sql(
    "SELECT context_type, COUNT(*) AS cached_rows, MAX(created_at) AS latest_cached_at "
    "FROM external_context_cache WHERE context_type IN ('weather_forecast','gbfs_station_status','mta_realtime') GROUP BY context_type"
)
display(feature_summary)
display(static_coverage)
display(cache_status)

/Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/7.13-7.18/db_utils.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn, params=params)
/Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/7.13-7.18/db_utils.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn, params=params)


,block,raw_features,model_features_after_encoding
0,learned baseline,6,12
1,venue-specific enrichment,15,venue_id + static attributes
2,dynamic serving context,13,rule/residual layer
3,final prediction inputs,28,enriched baseline + dynamic_delta


,feature,coverage_pct
0,day_of_week,100.00
1,hour_of_day,100.00
2,is_weekend,100.00
3,venue_type,100.00
4,district,100.00
5,rating,63.45
6,venue_id,100.00
7,latitude,100.00
8,longitude,100.00
9,weather_risk,100.00


,context_type,cached_rows,latest_cached_at
0,weather_forecast,5,2026-07-16 12:48:44
1,gbfs_station_status,8,2026-07-16 12:49:43
2,mta_realtime,7,2026-07-16 12:50:15


In [4]:
# 3. Train both temporal-snapshot baselines and generate the dynamically adjusted 12-hour curve.
subprocess.run([
    sys.executable, 'forecast_v2_pattern.py',
    '--labels', str(LABELS), '--legacy-labels', str(LEGACY_LABELS), '--output-dir', str(MODEL_DIR),
], cwd=HERE, check=True)

/Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/7.13-7.18/db_utils.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn, params=params)
/Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/7.13-7.18/db_utils.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn, params=params)


               model_name           model_variant                    split_type split  raw_feature_count  feature_count    mae   rmse    r2  mae_improvement_vs_baseline
GradientBoostingRegressor           baseline_time known_venue_temporal_snapshot train                  6             12 14.742 19.306 0.601                          NaN
GradientBoostingRegressor           baseline_time known_venue_temporal_snapshot  test                  6             12 14.766 19.332 0.600                        0.000
GradientBoostingRegressor venue_specific_enriched known_venue_temporal_snapshot train                 15            185 13.929 17.963 0.654                          NaN
GradientBoostingRegressor venue_specific_enriched known_venue_temporal_snapshot  test                 15            185 13.944 17.980 0.654                        0.822


/Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/7.13-7.18/db_utils.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn, params=params)


CompletedProcess(args=['/Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/.venv-1/bin/python', 'forecast_v2_pattern.py', '--labels', '/Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/7.13-7.18/output/serpapi_v2_labels_20260716/serpapi_popular_times_weak_labels.csv', '--legacy-labels', '/Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/7.13-7.18/output/serpapi_repeat_audit/legacy_cached_baseline.csv', '--output-dir', '/Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/7.13-7.18/output/v2_pattern_known_venue_serpapi_20260716'], returncode=0)

In [5]:
# 4. Review the same temporal split: baseline vs venue-specific enrichment.
metrics = pd.read_csv(MODEL_DIR / 'forecast_v2_pattern_model_metrics.csv')
curve = pd.read_csv(MODEL_DIR / 'prediction_curve_v2_pattern.csv')
manifest = json.loads((MODEL_DIR / 'forecast_v2_pattern_manifest.json').read_text())
display(metrics[metrics['split'].eq('test')])
display(curve[['baseline_score', 'dynamic_delta', 'predicted_score']].describe())
display(curve[['weather_context_available', 'gbfs_context_available', 'mta_context_available', 'dynamic_context_available']].mean().mul(100).rename('available_pct'))
display(curve['predicted_level'].value_counts().rename_axis('level').reset_index(name='rows'))
manifest

,model_name,model_variant,split_type,split,raw_feature_count,feature_count,mae,rmse,r2,mae_improvement_vs_baseline
1,GradientBoostingRegressor,baseline_time,known_venue_temporal_snapshot,test,6,12,14.766,19.332,0.600,0.000
3,GradientBoostingRegressor,venue_specific_enriched,known_venue_temporal_snapshot,test,15,185,13.944,17.980,0.654,0.822


,baseline_score,dynamic_delta,predicted_score
count,1944.000000,1944.000000,1944.000000
mean,27.162901,1.040000,28.202901
std,16.180580,0.866248,16.811509
min,0.000000,0.540000,0.540000
25%,16.220000,0.540000,16.760000
50%,26.190000,0.540000,26.730000
75%,36.262500,1.040000,37.737500
max,81.010000,2.540000,83.550000


weather_context_available    100.0
gbfs_context_available       100.0
mta_context_available        100.0
dynamic_context_available    100.0
Name: available_pct, dtype: float64

,level,rows
0,quiet,1191
1,moderate,729
2,busy,24


{'model_version': 'forecast-v2-known-venue-serpapi-context',
 'target_type': 'google_popular_times_proxy',
 'training_rows': 20151,
 'test_rows': 20157,
 'venues': 162,
 'places': 116,
 'split': 'known_venue_temporal_snapshot',
 'validation_status': 'not available: no third temporally complete SerpAPI cohort',
 'baseline_features': ['day_of_week',
  'hour_of_day',
  'is_weekend',
  'venue_type',
  'district',
  'rating'],
 'enriched_features': ['day_of_week',
  'hour_of_day',
  'is_weekend',
  'venue_type',
  'district',
  'rating',
  'venue_id',
  'latitude',
  'longitude',
  'weather_risk',
  'source_confidence',
  'accessible_status',
  'active_warning',
  'open_now',
  'rating_missing'],
 'excluded_low_coverage_features': ['borough',
  'opening_hours',
  'primary_language',
  'venue_accessibility',
  'venue_language'],
 'dynamic_context_columns': ['temperature_c',
  'precipitation_mm',
  'relative_humidity_pct',
  'wind_speed_kmh',
  'heat_alert',
  'citibike_station_activity',
  '